<a href="https://colab.research.google.com/github/rominaarndt/An-lisis-de-Datos-con-DuckDB-y-SQL/blob/main/An%C3%A1lisis_de_Datos_Meteorol%C3%B3gicos_con_DuckDB_y_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Temperature_anomaly_monthly.csv
Global + 7 regiones × 316 meses: anomalía de temperatura + ppm de CO2

In [6]:
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
    import duckdb

import sqlite3
import pandas as pd
print("✅ Listo.")


✅ Listo.


In [7]:
import duckdb

# Ejecutar la función read_csv_auto dentro de una consulta SQL de DuckDB
query = "SELECT * FROM read_csv_auto('https://raw.githubusercontent.com/rominaarndt/An-lisis-de-Datos-con-DuckDB-y-SQL/refs/heads/main/temperature_anomaly_monthly.csv') LIMIT 5"
duckdb.sql(query).show()

┌────────────┬───────┬───────┬─────────────────────┬────────────────┬───────────────────────┬─────────┐
│ year_month │ year  │ month │       region        │ temp_anomaly_c │ temp_anomaly_baseline │ co2_ppm │
│  varchar   │ int64 │ int64 │       varchar       │     double     │        varchar        │ double  │
├────────────┼───────┼───────┼─────────────────────┼────────────────┼───────────────────────┼─────────┤
│ 2000-01    │  2000 │     1 │ Global              │          0.432 │ 1951-1980             │  367.44 │
│ 2000-01    │  2000 │     1 │ Arctic              │            1.5 │ 1951-1980             │    NULL │
│ 2000-01    │  2000 │     1 │ Northern_Hemisphere │          0.468 │ 1951-1980             │    NULL │
│ 2000-01    │  2000 │     1 │ Southern_Hemisphere │           0.32 │ 1951-1980             │    NULL │
│ 2000-01    │  2000 │     1 │ Tropics             │          0.272 │ 1951-1980             │    NULL │
└────────────┴───────┴───────┴─────────────────────┴────────────

In [9]:
csv_path = "https://raw.githubusercontent.com/rominaarndt/An-lisis-de-Datos-con-DuckDB-y-SQL/refs/heads/main/temperature_anomaly_monthly.csv"

print("--- 1. Creación de vista virtual y Limpieza básica ---")
# DuckDB lee el archivo CSV directamente desde GitHub como una tabla relacional
duckdb.sql(f"""
    CREATE OR REPLACE VIEW clima AS
    SELECT
        year_month,
        year,
        month,
        region,
        temp_anomaly_c,
        temp_anomaly_baseline,
        COALESCE(co2_ppm, 0.0) AS co2_seguro -- Limpieza básica de nulos en co2_ppm
    FROM read_csv_auto('{csv_path}')
    WHERE temp_anomaly_c IS NOT NULL; -- Limpieza de nulos en temperaturas
""")

print("Vista 'clima' creada exitosamente.")

--- 1. Creación de vista virtual y Limpieza básica ---
Vista 'clima' creada exitosamente.


In [10]:
print("\n--- Consulta 1: Conteo total de registros y distribución por región ---")
conteo_df = duckdb.sql("""
    SELECT
        region,
        COUNT(*) AS total_registros,
        MIN(year) AS anio_inicio,
        MAX(year) AS anio_fin
    FROM clima
    GROUP BY region
    ORDER BY total_registros DESC;
""").df()
display(conteo_df)

print("\n--- Consulta 2: Promedios y extremos de anomalía térmica por región ---")
promedios_df = duckdb.sql("""
    SELECT
        region,
        ROUND(AVG(temp_anomaly_c), 4) AS temp_promedio,
        ROUND(MIN(temp_anomaly_c), 4) AS temp_minima,
        ROUND(MAX(temp_anomaly_c), 4) AS temp_maxima
    FROM clima
    GROUP BY region
    ORDER BY temp_promedio DESC;
""").df()
display(promedios_df)

print("\n--- Consulta 3: Filtrado complejo (Registros con anomalía superior al promedio global histórico) ---")
filtrado_df = duckdb.sql("""
    SELECT
        year_month,
        region,
        temp_anomaly_c
    FROM clima
    WHERE region = 'Global'
      AND temp_anomaly_c > (SELECT AVG(temp_anomaly_c) FROM clima WHERE region = 'Global')
    ORDER BY temp_anomaly_c DESC
    LIMIT 10;
""").df()
display(filtrado_df)


--- Consulta 1: Conteo total de registros y distribución por región ---


,region,total_registros,anio_inicio,anio_fin
0,Global,316,2000,2026
1,Arctic,316,2000,2026
2,North_America,316,2000,2026
3,Europe,316,2000,2026
4,Asia,316,2000,2026
5,Northern_Hemisphere,316,2000,2026
6,Southern_Hemisphere,316,2000,2026
7,Tropics,316,2000,2026



--- Consulta 2: Promedios y extremos de anomalía térmica por región ---


,region,temp_promedio,temp_minima,temp_maxima
0,Arctic,2.7253,1.500,4.522
1,Europe,1.2864,0.674,2.265
2,North_America,1.1335,0.616,2.009
3,Northern_Hemisphere,1.0159,0.468,1.793
4,Asia,0.9743,0.479,1.700
5,Global,0.7798,0.401,1.455
6,Southern_Hemisphere,0.5816,0.273,1.128
7,Tropics,0.5072,0.219,1.085



--- Consulta 3: Filtrado complejo (Registros con anomalía superior al promedio global histórico) ---


,year_month,region,temp_anomaly_c
0,2024-03,Global,1.455
1,2024-02,Global,1.452
2,2024-05,Global,1.435
3,2024-06,Global,1.407
4,2024-04,Global,1.372
5,2025-02,Global,1.355
6,2023-12,Global,1.350
7,2024-01,Global,1.314
8,2023-10,Global,1.309
9,2023-09,Global,1.285
